# 02 · Train the court keypoint model

A YOLO **pose** model that finds 18 court landmarks (baseline corners, lane corners, free-throw line
ends, half-court line ends). Four or more visible landmarks give the homography from the broadcast
image to the court in metres, which powers the tactical view, speed/distance and 3D reconstruction.

Keypoint order (must match `courtvision/court/geometry.py`):

| idx | landmark | idx | landmark |
|---|---|---|---|
| 0-5 | left baseline: corner, corner-3, lane, lane, corner-3, corner | 10-15 | right baseline (mirrored) |
| 6-7 | half-court line ends | 16-17 | right free-throw line ends |
| 8-9 | left free-throw line ends | | |

**Horizontal flip is turned off** (`fliplr=0`). Flipping an image turns the left baseline into the
right one, so unless `flip_idx` is set in `data.yaml` the model learns contradictory labels. The cell
below adds the correct `flip_idx`, after which you can turn flipping back on for more data.

**Zoomed, half-court views.** Every image in the public dataset shows the whole court centred in
frame (validation mAP50 0.995 says the model has learned exactly that), but on a real NBA clip it
saw only 5.2 of the 18 keypoints per frame. The crop cell below adds zoomed half-court crops of each
training and validation image, with the keypoints outside the crop marked invisible, so the model
learns the views it actually meets and validation measures them. Your own labelled broadcast
frames (from notebook 03) can be merged in the same way.

In [ ]:
# Runtime > Change runtime type > GPU before running.
!nvidia-smi -L
from google.colab import drive
drive.mount("/content/drive")

# Everything that must survive a Colab disconnect (runs, weights) lives in Drive.
DRIVE_ROOT = "/content/drive/MyDrive/courtvision"
import os
os.makedirs(f"{DRIVE_ROOT}/models", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)

In [ ]:
REPO_URL = "https://github.com/moorexavierb/CourtVision.git"
!git clone -q {REPO_URL} /content/courtvision || (cd /content/courtvision && git pull -q)
!pip -q install -e "/content/courtvision[models,train]"

In [ ]:
# Store your Roboflow key in Colab's Secrets panel (key icon on the left) as
# ROBOFLOW_API_KEY. Never paste it into a cell: notebooks get shared and committed.
from google.colab import userdata
ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

In [ ]:
# Public court-keypoint dataset (the one the reference repo uses). Add your own labelled frames
# from NBA broadcasts to this project in Roboflow for better results on your footage.
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace("fyp-3bwmg").project("reloc2-den7l").version(1).download(
    "yolov8", location="/content/datasets/court-keypoints")
DATA_YAML = f"{dataset.location}/data.yaml"
print(open(DATA_YAML).read())

In [ ]:
import yaml
cfg = yaml.safe_load(open(DATA_YAML))
# Mirror map: left baseline <-> right baseline, left FT <-> right FT; half-court ends map to themselves.
cfg["flip_idx"] = [15, 14, 13, 12, 11, 10, 6, 7, 16, 17, 5, 4, 3, 2, 1, 0, 8, 9]
yaml.safe_dump(cfg, open(DATA_YAML, "w"))
assert cfg["kpt_shape"][0] == 18, cfg["kpt_shape"]

Check `flip_idx` against your labels before trusting it: open a few training images with their
keypoints drawn (next cell). If the order differs, fix `flip_idx` or set `fliplr=0.0`.

In [ ]:
import cv2, glob, matplotlib.pyplot as plt, numpy as np
img_path = sorted(glob.glob(f"{dataset.location}/train/images/*"))[0]
lbl = img_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]
vals = np.array(open(lbl).readline().split()[5:], float).reshape(-1, 3)
for i, (x, y, v) in enumerate(vals):
    if v > 0:
        cv2.circle(img, (int(x * w), int(y * h)), 6, (255, 0, 0), -1)
        cv2.putText(img, str(i), (int(x * w) + 6, int(y * h) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
plt.figure(figsize=(14, 8)); plt.imshow(img); plt.axis("off")

## Zoomed half-court crops

Two crops per image at 1.6x to 3x zoom, biased toward one basket like a half-court broadcast view.
Crops with fewer than 4 visible keypoints are skipped (the homography needs 4). Validation images
get crops too, so the score below reflects zoomed views and not only full-court ones. Run once per
download; re-running skips images that are already crops.

In [ ]:
from courtvision.training.court_crops import make_zoomed_crops
for split, seed in [("train", 0), ("valid", 1)]:
    n = make_zoomed_crops(dataset.location, split=split, per_image=2, zoom_range=(1.6, 3.0), seed=seed)
    print(f"{split}: added {n} zoomed crops")

In [ ]:
# Optional: add your own labelled broadcast frames (notebook 03, corrected in Roboflow, YOLO export).
# They go in after the crops so they are used as they are. Keep the keypoint order of the table above.
import shutil
OWN = f"{DRIVE_ROOT}/training_data/own_labelled_court"
if os.path.isdir(OWN):
    for split in ["train", "valid"]:
        for kind in ["images", "labels"]:
            for f in glob.glob(f"{OWN}/{split}/{kind}/*"):
                shutil.copy(f, f"{dataset.location}/{split}/{kind}/own_{os.path.basename(f)}")
    print("added", len(glob.glob(f"{dataset.location}/train/images/own_*")), "own training frames")

In [ ]:
# Look at a few crops with their labels before training.
crops = sorted(glob.glob(f"{dataset.location}/train/images/*_zoom*"))[:3]
fig, axes = plt.subplots(1, len(crops), figsize=(18, 4))
for ax, p in zip(np.atleast_1d(axes), crops):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    v = np.array(open(p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt").readline().split()[5:], float).reshape(-1, 3)
    for i, (x, y, vis) in enumerate(v):
        if vis > 0:
            cv2.circle(im, (int(x * w), int(y * h)), 6, (255, 0, 0), -1)
            cv2.putText(im, str(i), (int(x * w) + 6, int(y * h) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
    ax.imshow(im); ax.axis("off")

In [ ]:
import os
from ultralytics import YOLO
RUN_NAME = "court_kp_yolo11m_pose_zoom"  # new run: the old one finished and would not resume
last = f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/last.pt"
resume = os.path.exists(last)
model = YOLO(last if resume else "yolo11m-pose.pt")
model.train(data=DATA_YAML, imgsz=640, epochs=200, patience=40, batch=-1, cos_lr=True,
            fliplr=0.5, mosaic=0.5, degrees=0, perspective=0.0005, scale=0.3, save_period=10,
            project=f"{DRIVE_ROOT}/runs", name=RUN_NAME, exist_ok=True, resume=resume)

In [ ]:
import shutil
best = f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/best.pt"
print(YOLO(best).val(data=DATA_YAML, imgsz=640).pose.map50)
shutil.copy(best, f"{DRIVE_ROOT}/models/court_keypoints.pt")